# Priority-Flood: conditioning a digital elevation model

Explore how a depression is raised to its lowest spill elevation. The lab preserves the original DEM, displays the filled surface and makes the fill-depth layer visible.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/priority-flood'))
from priority_flood import priority_flood


## 1. A synthetic depression

The right edge is a lower boundary outlet. The bowl's centre can spill east through the lower saddle; the algorithm should raise the centre to the lowest reachable spill level.

In [ ]:
dem = np.array([
    [9, 9, 9, 9, 7],
    [9, 8, 8, 8, 7],
    [9, 8, 2, 7, 7],
    [9, 8, 8, 8, 7],
    [9, 9, 9, 9, 7],
] , dtype=float)
filled, depth = priority_flood(dem)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(axes, [dem, filled, depth], ['Original DEM', 'Filled DEM', 'Fill depth']):
    im = ax.imshow(data, cmap='terrain' if title != 'Fill depth' else 'magma')
    ax.set_title(title); ax.set_xticks(range(dem.shape[1])); ax.set_yticks(range(dem.shape[0]))
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
print('Centre before/after/depth:', dem[2, 2], filled[2, 2], depth[2, 2])


## 2. NoData is a boundary decision

This implementation treats valid cells adjacent to NoData as outlet seeds. That can be appropriate for a known drainage boundary, but an internal data gap may not be a real outlet.

In [ ]:
dem_with_gap = dem.copy()
dem_with_gap[2, 2] = np.nan
filled_gap, depth_gap = priority_flood(dem_with_gap)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(np.ma.masked_invalid(filled_gap), cmap='terrain')
ax.set_title('Filled surface with an internal NoData gap')
fig.colorbar(im, ax=ax, label='Elevation'); plt.show()


## Investigate

1. Change the boundary outlet elevation. How does the pit's spill level change?
2. Compare the original and filled DEM. Where did the algorithm alter the surface?
3. Why is the fill-depth raster important to retain?
4. What changes if an internal NoData gap is treated as an outlet?
5. Why does filling not automatically resolve all flats for D8 routing?
6. When might breaching be preferable to filling?